# E14 — Label-Noise Sensitivity Analysis (Contribution 2) — Gate 3

**Experiment ID:** `E14`. **Spec:** `EXPERIMENT_PLAN.md` §E14. **Governing rules:** `CLAUDE.md`.
Phase 4, its own single-experiment batch. Execution **stops at the Gate 3 boundary**; nothing in
Phase 5 (E15-E18) runs here.

**Objective.** Quantify how known covariance miscalibration in the risk labels propagates into the
coverage validity established at Gate 2.

**Protocol.** Every choice below is fixed by the **2026-09-16 E14 PRE-REGISTRATION** entry in
`DECISIONS.md`, written *before* this notebook read the official test set (CLAUDE.md §3). The
binding scope is **SCOPED M7**, from the **2026-09-16 Assumption-A4 / Q-LBL-01** resolution
(A4 = PARTIAL HOLD; Q-LBL-01 = option (b)).

* **Scaling semantics (Q-LBL-02 (a)):** one scalar on the **combined** position covariance,
  `C -> s*C`. A *variance* scale, so sigmas scale by `sqrt(s)`. Grid `{0.8 … 2.0}`, anchored at 1.0.
* **Evaluation-only (Q-LBL-03):** no model is refit; conformal calibration keeps its **original**
  labels. Only official-test labels are regenerated.
* **Two label arms, both declared in advance.** **(P) direct** — the label regenerated outright;
  **(A) anchored** — the reported label displaced by exactly the shift rescaling induces, which
  cancels E3's recomputation discrepancy. Neither is privileged as "the truth".
* **Populations fixed in advance.** All M7-eligible supported events, and — the **primary
  interpretive focus** — the **high-risk stratum defined by the ORIGINAL reported label**, so the
  same events are compared at every grid point.
* **Trend statistics are DESCRIPTIVE.** Q-STAT-04 spends the project's single formal confirmatory
  contrast on E10-vs-E11; no p-value here is a confirmatory test.

Results are reported exactly as observed (CLAUDE.md §3, §9). **No gate call is made in this
notebook** — Gate 3 (venue tier) is Sidh's.

In [ ]:
# --- Setup + provenance (invariant I4) ------------------------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import labelnoise_runner as LN
from kelvins_conformal.reporting import write_table_atomic

cfg = load_config()
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git","rev-parse","HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{name}.csv"); print(f"saved: reports/tables/{name}.csv")

def save_fig(fig, name):
    for ext in ("png","pdf"): fig.savefig(FIGDIR/f"{name}.{ext}", dpi=160, bbox_inches="tight")
    print(f"saved: reports/figures/{name}.png|pdf")

PROVENANCE = {"experiment_ids": ["E14"], "git_commit_sha": git_sha(),
              "config_hash": cfg.config_hash, "seeds": list(cfg.train.seeds),
              "scaling_grid": list(cfg.labelnoise.scaling_grid),
              "bootstrap_resamples": cfg.bootstrap.n_resamples,
              "executed_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0]}
print(json.dumps(PROVENANCE, indent=2))
CDIR, CANC, CREF = "#0072B2", "#009E73", "#D55E00"

## 1. Run E14

In [ ]:
RES = LN.run_e14(cfg)
meta = RES["meta"]
print("seeds:", meta["seeds"], "| levels:", meta["levels"], "| grid:", meta["scaling_grid"])
print("methods:", meta["methods"], "| learners:", meta["learners"])
cov = RES["coverage"]
for key in ("coverage", "representativeness", "anchor_agreement", "failure_check", "eligibility"):
    save_table(RES[key], f"e14_{key}")
save_table(RES["trend"], "e14_trend")

## 2. Scope of the M7 subset, and the pre-registered failure criterion

`EXPERIMENT_PLAN.md` E14's failure criterion — "M7 subset too small or unrepresentative to support
any claim" — was instantiated numerically *before* the run (pre-registration §11). Its verdict is
read off here rather than judged after the fact.

In [ ]:
display(RES["eligibility"].T)
print("\n=== pre-registered failure criterion ===")
display(RES["failure_check"])
FAILED = bool(RES["failure_check"].set_index("criterion").loc["E14_FAILURE_CRITERION_MET","fires"])
print(f"E14 failure criterion met: {FAILED}")
print("(If True, the Q-LBL-01 option-(c) fallback is SURFACED to Sidh, not adopted here.)")

## 3. Representativeness: M7-eligible subset vs. the full official test set

Mandated by the Assumption-A4 resolution: *a scope restriction that is not quantified is an
undisclosed change of estimand.* Named table `e14_m7_representativeness`, named figure of the
same stem.

In [ ]:
rep = RES["representativeness"]
display(rep.round(4))
save_table(rep, "e14_m7_representativeness")
print(f"KS statistic = {rep['ks_statistic'].iloc[0]:.4g}, p = {rep['ks_pvalue'].iloc[0]:.4g}")

elig = RES["eligibility"].iloc[0]
frac = elig["frac_m7_eligible"]
print(f"M7-eligible: {int(elig['n_m7_eligible'])} / {int(elig['n_official_test'])} = {frac:.4%}")

fig, ax = plt.subplots(figsize=(6.6,4))
lab_full = RES["labels"]
# ECDF of the reported label for both populations (identical when eligibility is total).
from kelvins_conformal.data import official_test_target_cdms
from kelvins_conformal.labelnoise import rescale as _rs
_el = _rs.eligibility(official_test_target_cdms(cfg))
for name, y, c in (("full official test", _el["target_log_risk"].to_numpy(float), CREF),
                   ("M7-eligible subset",
                    _el.loc[_el.m7_eligible,"target_log_risk"].to_numpy(float), CDIR)):
    xs = np.sort(y); ys = np.arange(1, xs.size+1)/xs.size
    ax.step(xs, ys, where="post", label=f"{name} (n={xs.size})", color=c,
            lw=2.4 if "full" in name else 1.4, alpha=0.85,
            ls="-" if "full" in name else "--")
ax.axvline(cfg.high_risk_threshold, color="grey", ls=":", lw=1.2, label="high-risk threshold")
ax.set_xlabel("reported label  (log10 risk)"); ax.set_ylabel("ECDF")
ax.set_title("E14 representativeness: M7-eligible subset vs full official test set")
ax.legend(frameon=False, loc="upper left", fontsize=9)
save_fig(fig, "e14_m7_representativeness")
plt.show()

## 4. Anchor agreement (s = 1.0) — extending E3's A4 measurement to the full subset

E3 measured recomputed-vs-reported agreement on a **100-CDM stratified sample**; the same
comparison over **every** M7-eligible official-test event says whether the PARTIAL HOLD recorded
for Assumption A4 also describes the population E14 actually uses. Strata are the pre-declared
`pc_spike.risk_strata_edges` bands, reused rather than re-cut.

In [ ]:
ag = RES["anchor_agreement"]
display(ag.round(4))
tol = cfg.pc_spike.tolerance.abs_log10_risk
print(f"(tolerance = +/-{tol} log10; E3's 100-CDM sample gave 65.0% within it overall,")
print(" and was BEST in the (-6, 0] high-risk band — the basis for the scoped-M7 decision.)")

## 5. Headline figure — coverage vs. covariance-scaling factor

The spec's headline deliverable: coverage against the scaling factor, with CI bands and the
nominal-coverage reference line. The **primary** curve is `E11_weighted_rule` (the Gate-2 method)
on **persistence** (the learner carrying the pre-registered primary contrast), two-sided, at the
primary nominal level.

In [ ]:
prim = meta["primary_level"]

def curve(arm, population, method, learner, nominal=None):
    nominal = prim if nominal is None else nominal
    q = cov[(cov.arm==arm)&(cov.population==population)&(cov.method==method)
            &(cov.learner==learner)&(cov.nominal==nominal)]
    return q.sort_values("scale")

def plot_panel(ax, population, arm, nominal, title):
    for method, lrn, c, mk in (("E11_weighted_rule","persistence",CDIR,"o"),
                               ("E10_naive_official","persistence",CREF,"s"),
                               ("E12_cqr_weighted_rule","gbm",CANC,"^")):
        g = curve(arm, population, method, lrn, nominal)
        if g.empty: continue
        ax.plot(g.scale, g.coverage_mean, marker=mk, color=c, lw=1.8,
                label=f"{method} ({lrn})")
        ax.fill_between(g.scale, g.cp_lo_mean, g.cp_hi_mean, color=c, alpha=0.15, lw=0)
    ax.axhline(nominal, color="k", ls="--", lw=1.2, label=f"nominal {nominal:.0%}")
    ax.axvline(1.0, color="grey", ls=":", lw=1.0)
    ax.set_xlabel("covariance-scaling factor  s"); ax.set_ylabel("empirical coverage")
    ax.set_title(title, fontsize=10)

fig, axes = plt.subplots(1, 2, figsize=(12.4,4.4), sharey=True)
plot_panel(axes[0], "m7_all", "anchored", prim,
           f"All M7-eligible events (anchored labels), nominal {prim:.0%}")
plot_panel(axes[1], "m7_high_risk", "anchored", prim,
           f"PRIMARY FOCUS: high-risk stratum (anchored), nominal {prim:.0%}")
axes[1].legend(frameon=False, fontsize=8, loc="best")
fig.suptitle("E14 headline: coverage vs covariance-scaling factor (CP CI bands)", y=1.02)
save_fig(fig, "e14_headline_coverage_vs_scaling")
plt.show()

### 5b. The same curves under the **direct** label arm

The direct arm regenerates the label outright, so its *level* also carries E3's recomputation
discrepancy (A4 is only a PARTIAL HOLD). Both arms were declared before either was computed; both
are shown.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.4,4.4), sharey=True)
plot_panel(axes[0], "m7_all", "direct", prim,
           f"All M7-eligible events (direct labels), nominal {prim:.0%}")
plot_panel(axes[1], "m7_high_risk", "direct", prim,
           f"High-risk stratum (direct labels), nominal {prim:.0%}")
axes[1].legend(frameon=False, fontsize=8, loc="best")
fig.suptitle("E14: coverage vs scaling factor, DIRECT label arm", y=1.02)
save_fig(fig, "e14_coverage_vs_scaling_direct")
plt.show()

## 6. Scaling-factor sensitivity table

The spec's first named table. Reported for the primary method/learner at every nominal level, both
arms, both populations.

In [ ]:
sens = cov[(cov.method=="E11_weighted_rule")&(cov.learner=="persistence")].copy()
sens = sens[["arm","population","nominal","scale","coverage_mean","coverage_sd",
             "cp_lo_mean","cp_hi_mean","gap_pp","n","n_seeds"]].sort_values(
    ["arm","population","nominal","scale"])
display(sens.round(4))
save_table(sens, "e14_scaling_sensitivity")

print("\nPrimary method/learner/level, high-risk stratum:")
for arm in ("anchored","direct"):
    g = curve(arm, "m7_high_risk", "E11_weighted_rule", "persistence")
    line = "  ".join(f"s={r.scale:.1f}: {r.coverage_mean:.3f}" for r in g.itertuples())
    print(f"  [{arm:8s}] {line}   (n={int(g.n.iloc[0])})")

## 7. Trend statistics — DESCRIPTIVE (Q-STAT-04)

OLS slope of coverage on `s` with a 95% CI, plus Spearman rho for monotonicity. `EXPERIMENT_PLAN`
E14 names a "trend test"; the Q-STAT-04 resolution reserves formal confirmatory testing for the
single E10-vs-E11 contrast. E14 therefore computes exactly the named quantities and reports them
**with CIs, labelled exploratory** — the conflict is flagged, not silently resolved.

In [ ]:
tr = RES["trend"]
main = tr[(tr.method=="E11_weighted_rule")&(tr.learner=="persistence")&(tr.nominal==prim)]
display(main.round(4))
print("\nAll methods, primary level, high-risk stratum:")
display(tr[(tr.nominal==prim)&(tr.population=="m7_high_risk")].round(4))

## 8. Degradation-rate summary figure (all methods, both populations)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12.4,8), sharex=True, sharey="row")
for i, pop in enumerate(("m7_all","m7_high_risk")):
    for j, arm in enumerate(("anchored","direct")):
        ax = axes[i][j]
        for lvl, c in zip(meta["levels"], (CDIR, CANC, CREF)):
            g = curve(arm, pop, "E11_weighted_rule", "persistence", lvl)
            if g.empty: continue
            ax.plot(g.scale, 100*(g.coverage_mean-g.nominal), marker="o", color=c, lw=1.7,
                    label=f"nominal {lvl:.0%}")
        ax.axhline(0.0, color="k", ls="--", lw=1.1); ax.axvline(1.0, color="grey", ls=":", lw=1.0)
        ax.set_title(f"{pop} / {arm}", fontsize=10)
        if i==1: ax.set_xlabel("covariance-scaling factor  s")
        if j==0: ax.set_ylabel("coverage gap vs nominal (pp)")
axes[0][1].legend(frameon=False, fontsize=9)
fig.suptitle("E14: coverage gap vs scaling factor — E11 weighted conformal (persistence)", y=1.0)
save_fig(fig, "e14_degradation_by_level")
plt.show()

## 9. E14 findings — measurement only, reported exactly as observed

In [ ]:
def at(arm, pop, method, lrn, s, nominal=None):
    g = curve(arm, pop, method, lrn, nominal)
    r = g[np.isclose(g.scale, s)]
    return (float(r.coverage_mean.iloc[0]), float(r.cp_lo_mean.iloc[0]), float(r.cp_hi_mean.iloc[0]))

grid = meta["scaling_grid"]
smin, smax = min(grid), max(grid)
e = RES["eligibility"].iloc[0]
m = main.iloc[0] if len(main) else None
hr_anch = curve("anchored","m7_high_risk","E11_weighted_rule","persistence")
all_anch = curve("anchored","m7_all","E11_weighted_rule","persistence")
tr_hr = tr[(tr.arm=="anchored")&(tr.population=="m7_high_risk")&(tr.method=="E11_weighted_rule")
           &(tr.learner=="persistence")&(tr.nominal==prim)].iloc[0]
tr_all = tr[(tr.arm=="anchored")&(tr.population=="m7_all")&(tr.method=="E11_weighted_rule")
            &(tr.learner=="persistence")&(tr.nominal==prim)].iloc[0]

print(f"""
E14 / LABEL-NOISE SENSITIVITY — WHAT THE BATCH SHOWS (measurement only, exactly as observed)

 SCOPE ACTUALLY REALISED:
  * M7-eligible: {int(e['n_m7_eligible'])} / {int(e['n_official_test'])}
    ({e['frac_m7_eligible']:.2%}) of official-test events. Analysis population
    {int(e['n_analysis_anchored'])} (anchored) / {int(e['n_analysis_direct'])} (direct);
    high-risk focus stratum n = {int(e['n_analysis_anchored_high_risk'])}.
  * Pre-registered failure criterion met: {FAILED}.

 PRIMARY CURVE (E11 weighted conformal, persistence, nominal {prim:.0%}, anchored labels,
 high-risk focus stratum, n = {int(hr_anch.n.iloc[0])}):
  * s = {smin}: {at('anchored','m7_high_risk','E11_weighted_rule','persistence',smin)[0]:.3f}
    | s = 1.0: {at('anchored','m7_high_risk','E11_weighted_rule','persistence',1.0)[0]:.3f}
    | s = {smax}: {at('anchored','m7_high_risk','E11_weighted_rule','persistence',smax)[0]:.3f}
  * OLS slope = {tr_hr.slope_per_unit_scale:+.4f} per unit s
    (95% CI [{tr_hr.slope_lo:+.4f}, {tr_hr.slope_hi:+.4f}]); Spearman rho = {tr_hr.spearman_rho:+.3f};
    coverage range across the grid = {tr_hr.coverage_range_pp:.2f} pp. DESCRIPTIVE (Q-STAT-04).

 WHOLE M7 SUBSET (same method/level/arm, n = {int(all_anch.n.iloc[0])}):
  * s = {smin}: {at('anchored','m7_all','E11_weighted_rule','persistence',smin)[0]:.3f}
    | s = 1.0: {at('anchored','m7_all','E11_weighted_rule','persistence',1.0)[0]:.3f}
    | s = {smax}: {at('anchored','m7_all','E11_weighted_rule','persistence',smax)[0]:.3f}
  * OLS slope = {tr_all.slope_per_unit_scale:+.4f} (95% CI [{tr_all.slope_lo:+.4f}, {tr_all.slope_hi:+.4f}]);
    Spearman rho = {tr_all.spearman_rho:+.3f}; range = {tr_all.coverage_range_pp:.2f} pp.

 NOT DECIDED HERE (CLAUDE.md §3, §9, §10, §13.7): the Gate 3 venue-tier call; whether
 Contribution 2 is established; whether any fallback is adopted. Execution stops at the Gate 3
 boundary — no Phase 5 (E15-E18) work is done in this notebook.
""")
(cfg.path("reports_dir")/"04_labelnoise_provenance.json").write_text(json.dumps(PROVENANCE, indent=2), encoding="utf-8")
print("provenance:", cfg.path("reports_dir")/"04_labelnoise_provenance.json")